## 1. Extracción y visualización de Datos
En esta primera etapa, se importan las librerías de alto rendimiento (Polars) para la lectura de más de 350,000 registros meteoceanográficos de Sabancuy, Campeche. Se realiza el proceso de limpieza de datos (*Data Cleaning*), que incluye el casteo de variables numéricas, el filtrado de valores nulos y la creación de una variable temporal unificada (`Fecha_Hora`). 

In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import seaborn as sns
from scipy.stats import gumbel_r
import pandas as pd

# Load data
BASE = Path().resolve().parent  # Sube de Notebook/ a src/
df = pl.read_excel(BASE / "data" / "DataSetLimpioSabancuy.xlsx")
#casting for year, month, day and hour to int for better handling of data
df = df.with_columns([
    pl.col(c).cast(pl.Int32) for c in ["Mes", "Dia", "Hora", "Anio"]
])
# create a new column for datetime using the year, month, day and hour columns
df= df.with_columns(
  pl.datetime(
    year=pl.col("Anio"),
    month=pl.col("Mes"),
    day=pl.col("Dia"),
    hour=pl.col("Hora")
  ).alias("Fecha_Hora")
)
print(df.head(10))

## 2. Análisis Estadístico Descriptivo
En esta sección se exploran a profundidad las características de las variables clave mediante herramientas visuales y estadísticas. En primer lugar, se evalúa la distribución y dispersión de los datos empleando histogramas de frecuencia (optimizados con la regla de Freedman-Diaconis para manejar el gran volumen de registros). Posteriormente, se analiza la estacionalidad del clima marítimo a través de gráficos de barras con promedios mensuales, facilitando la identificación de temporadas de alta energía. Por último, se presenta una matriz de correlación (mapa de calor) para cuantificar la relación matemática entre las variables meteoceanográficas, destacando la dependencia directa entre la velocidad del viento y la altura del oleaje.

In [ ]:
# Summary statistics for the main variables
summary = df.select([
  "VelocidadViento",
  "DireccionViento",
  "AlturaDelOleaje",
  "DireccionOleaje",
  "PeriodoDeLaOla",
]).describe()
print(summary)
# histogram of the wave height distribution using freedman-diaconis rule for bin width
wave_height = df["AlturaDelOleaje"].drop_nulls().to_numpy()
counts, bin_edges= np.histogram(wave_height, bins="fd")
number_of_bins = len(bin_edges) - 1
print(f"Number of bins: {number_of_bins}")
fig = px.histogram(
    df.to_pandas(),
    x="AlturaDelOleaje",
    nbins=number_of_bins,
    title="Distribución de la Altura del Oleaje en Sabancuy",
    labels={"AlturaDelOleaje": "Altura del Oleaje (m)"}
)
fig.show()

In [10]:
# Mean wind speed by month
mean_wind_speed = df.group_by("Mes").agg([
    pl.col("VelocidadViento").mean().alias("VelocidadViento_Media"),
    pl.col("VelocidadViento").std().alias("VelocidadViento_Std")
]).sort("Mes")

# Bar plot
fig_mean_wind = px.bar(
  mean_wind_speed.to_pandas(), # convert to pandas for plotly
  x="Mes",
  y="VelocidadViento_Media",
  title="Velocidad Media del Viento por Mes",
  labels={"Mes": "Mes", "VelocidadViento_Media": "Velocidad Media del Viento (m/s)"},
  color="VelocidadViento_Media",
  color_continuous_scale="Viridis" # color scale for better visualization
)
fig_mean_wind.update_xaxes(tickvals=list(range(1, 13)), ticktext=["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"])
fig_mean_wind.show()

In [ ]:
# correlation matrix for wind weight and wave height
correlation_weight_wave = df.select(
  ["VelocidadViento", "AlturaDelOleaje"]
).to_pandas().corr()
correlation_weight_fig= px.imshow(
  correlation_weight_wave,
  text_auto=True,
  aspect="auto",
  title="Correlación entre Velocidad del Viento y Altura del Oleaje",
  color_continuous_scale="RdBu_r"
)
correlation_weight_fig.show()
  

## 3. Distribuciones de Probabilidad y Ajustes Teóricos
En esta sección se analiza la distribución de los datos de oleaje para el diseño de infraestructura costera. Se utilizan histogramas de frecuencia absoluta ajustados a una distribución de valores extremos (Gumbel), integrando visualmente los estadísticos descriptivos (media y desviación estándar). Además, se presentan las Curvas de Probabilidad de Excedencia, las cuales son fundamentales en la ingeniería marítima para determinar el porcentaje de tiempo que una variable supera un umbral de diseño específico operativo o de supervivencia.

In [ ]:
# scatter plot of wave height vs wave period
df_scatter = df.drop_nulls(subset=["AlturaDelOleaje", "PeriodoDeLaOla"]) # drop rows with null values in the relevant columns
# choosing a sample of 10% of the data for better visualization
df_sample = df_scatter.sample(fraction=0.05, seed=42)
print(f"Original data size: {len(df_scatter)}, Sample size: {len(df_sample)}") 
scatter_fig = px.scatter(
  df_sample.to_pandas(),
  x="PeriodoDeLaOla",
  y="AlturaDelOleaje",
  title="Relación entre Periodo de la Ola y Altura del Oleaje",
  labels={"PeriodoDeLaOla": "Periodo de la Ola (s)", "AlturaDelOleaje": "Altura del Oleaje (m)"},
  opacity=0.5, # make points semi-transparent for better visibility
  color="VelocidadViento", # color points by wind speed for additional insight
  color_continuous_scale="Viridis"
)
scatter_fig.show()

## 4. Análisis Direccional: Rosas de Viento y Oleaje
Caracterización espacial de la energía marina mediante representaciones polares. Los datos se agrupan en 32 sectores direccionales de 11.25° cada uno, aplicando un desfase trigonométrico (-5.625°) para garantizar el centrado exacto sobre el eje Norte (0°). Estas gráficas permiten identificar visualmente la dirección predominante de los vientos más intensos y confirmar de qué cuadrante proviene la mayor carga energética del oleaje que incide sobre la costa.

In [ ]:
# Time series of wave height over time
df_ordened = df.sort("Fecha_Hora").drop_nulls("AlturaDelOleaje")
# extract the date and wave height as numpy arrays for plotting
dates = df_ordened["Fecha_Hora"].to_numpy()
wave_heights = df_ordened["AlturaDelOleaje"].to_numpy()
# create a line plot of wave height over time
time_series_fig = go.Figure()
time_series_fig.add_trace(go.Scatter(
  x=dates,
  y=wave_heights,
  mode="lines",
  line=dict(color="blue", width=.5),
  name="Altura del Oleaje"
))

time_series_fig.update_layout(
  title=dict(
    text="<b> Registro Oleaje - Tiempo </b> <br>Sabancuy, Campeche <i> 1979 - 2018 </i>",
    x=0.5, # center the title
    xanchor="center",
    font=dict(size=18)
  ),
  xaxis_title="Tiempo [Años]",
  yaxis_title="Altura del Oleaje [m]",
  plot_bgcolor="white",
  yaxis=dict(
    showgrid=True,
    gridcolor="lightgrey",
    zeroline=True,
    zerolinecolor="lightgrey"
),
  xaxis=dict(
    showgrid=False,
    tickformat="%Y",
    dtick="M48"
),
  margin=dict(l=60, r=60, t=80, b=60)
  )
time_series_fig.show()

## 4.Rosas de Oleaje y Viento

In [ ]:
# Definir los límites y etiquetas (Asegúrate de que coincidan exactamente con el diccionario de colores)
bins_speed = [0, 2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5, 20.0, 22.5, 25.0, 27.5, 30.0]
speed_labels = [
    "<=2.5", ">2.5 - 5", ">5 - 7.5", ">7.5 - 10", ">10 - 12.5", 
    ">12.5 - 15", ">15 - 17.5", ">17.5 - 20", ">20 - 22.5", 
    ">22.5 - 25", ">25 - 27.5", ">27.5 - 30"
]

# 32 directions (360° / 32 = 11.25° per sector)
bins_direction = np.arange(-5.625, 360, 11.25)
labels_direction = np.arange(0, 360, 11.25)

# Extraer a pandas
df_pandas = df.select(["VelocidadViento", "DireccionViento"]).drop_nulls().to_pandas()

# Crear columna de rango de velocidad
df_pandas['Rango_Velocidad'] = pd.cut(df_pandas['VelocidadViento'], bins=bins_speed, labels=speed_labels)

# Clasificar direcciones y crear la nueva columna 'Direccion_Sector'
index_dir = np.digitize(df_pandas["DireccionViento"], bins=bins_direction)
secure_Index= (index_dir - 1) % len(labels_direction) 
df_pandas["Direccion_Sector"] = np.where(index_dir == 33, 0, labels_direction[secure_Index])

# Calcular porcentajes (Nota los corchetes [] dentro del groupby)
df_grouped = df_pandas.groupby(["Direccion_Sector", "Rango_Velocidad"], observed=False).size().reset_index(name="Count")
data_total = len(df_pandas)
df_grouped["Percentage"] = (df_grouped["Count"] / data_total) * 100

# Diccionario de colores (Las llaves ahora coinciden exactamente con speed_labels)
colores_rosa = {
    "<=2.5": "#ffffcc", ">2.5 - 5": "#ffff00", ">5 - 7.5": "#ffcc66", 
    ">7.5 - 10": "#ff9900", ">10 - 12.5": "#ff6666", ">12.5 - 15": "#e60000", 
    ">15 - 17.5": "#b30000", ">17.5 - 20": "#990000", ">20 - 22.5": "#800000", 
    ">22.5 - 25": "#663300", ">25 - 27.5": "#4d4d4d", ">27.5 - 30": "#000000"
}

# Crear gráfica
fig_wind_rose = px.bar_polar(
    df_grouped,
    r="Percentage",
    theta="Direccion_Sector",
    color="Rango_Velocidad",
    color_discrete_map=colores_rosa,
    template="plotly_white",
    title="<b>Rosa de Viento de Sabancuy</b>",
    labels={"Rango_Velocidad": "Velocidad (m/s)"}
)

# Ajustar formato
cleanest_labels = np.arange( 0, 360, 22.5)
fig_wind_rose.update_layout(
    polar=dict(
        radialaxis=dict(
            ticksuffix="%",
            angle=90,
            showticklabels=True
        ),
        angularaxis=dict(    # angularaxis VA DENTRO de polar
            direction="clockwise",
            rotation=90,
            tickmode="array",
            tickvals=cleanest_labels,
            ticktext=[str(v) if v != 0 else "0" for v in cleanest_labels],
            showticklabels=True,
            tickfont=dict(size=12)
        )
    ),
    legend=dict(
        orientation="v",
        yanchor="top", y=1.0,
        xanchor="left", x=1.1,
        bordercolor="lightgrey", 
        borderwidth=1,
        title_text="Rosa de Viento de Sabancuy",
        font=dict(size=11)
    ),
    margin=dict(l=80, r=220, t=100, b=80),
    font=dict(size=12)
)

fig_wind_rose.show()

In [ ]:
# 1. Definir los límites y etiquetas de Altura del Oleaje según la leyenda de tu imagen
bins_oleaje = [0, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 2.0, 2.4, 5.0, 10.0]
labels_oleaje = [
    "<=0.4", ">0.4 - 0.6", ">0.4 - 0.8", ">0.8 - 1", ">0.8 - 1.2", 
    ">1.2 - 1.4", ">1.2 - 1.6", ">1.6 - 2", ">2 - 2.4", ">2.4 - 5", ">5 - 10"
]
# Nota: La leyenda en tu imagen tiene un error tipográfico original (repite >0.4 y >0.8 de formas raras).
# He estructurado los rangos de forma lógica matemática para que no haya huecos en los datos.

# 2. Las 32 direcciones se mantienen igual (11.25° por sector)
bins_direction = np.arange(-5.625, 360, 11.25)
labels_direction = np.arange(0, 360, 11.25)

# 3. Extraer a pandas (AHORA USAMOS OLEAJE)
df_oleaje = df.select(["AlturaDelOleaje", "DireccionOleaje"]).drop_nulls().to_pandas()

# 4. Crear columna de rango de oleaje
df_oleaje['Rango_Oleaje'] = pd.cut(df_oleaje['AlturaDelOleaje'], bins=bins_oleaje, labels=labels_oleaje)

# 5. Clasificar direcciones del oleaje
index_dir = np.digitize(df_oleaje["DireccionOleaje"], bins=bins_direction)
secure_Index = (index_dir - 1) % len(labels_direction) 
df_oleaje["Direccion_Sector"] = labels_direction[secure_Index]

# 6. Calcular porcentajes
df_grouped_oleaje = df_oleaje.groupby(["Direccion_Sector", "Rango_Oleaje"], observed=False).size().reset_index(name="Count")
data_total_oleaje = len(df_oleaje)
df_grouped_oleaje["Percentage"] = (df_grouped_oleaje["Count"] / data_total_oleaje) * 100

# 7. Diccionario de colores (imitando la paleta azul-verde-amarillo de tu imagen)
colores_oleaje = {
    "<=0.4": "#2C105C",      # Azul muy oscuro/morado
    ">0.4 - 0.6": "#1E4B9C", 
    ">0.4 - 0.8": "#1965B0", 
    ">0.8 - 1": "#0083C5",   
    ">0.8 - 1.2": "#00A2D8", 
    ">1.2 - 1.4": "#4DBA9A", # Verdes
    ">1.2 - 1.6": "#1AB288", 
    ">1.6 - 2": "#76C759",   
    ">2 - 2.4": "#C7D44F",   # Amarillos/mostaza
    ">2.4 - 5": "#F2BB46",   
    ">5 - 10": "#F5E025"     # Amarillo brillante
}

# 8. Crear gráfica
fig_wave_rose = px.bar_polar(
    df_grouped_oleaje,
    r="Percentage",
    theta="Direccion_Sector",
    color="Rango_Oleaje",
    color_discrete_map=colores_oleaje,
    template="plotly_white",
    title="<b>Oleaje en Sabancuy</b>",
    labels={"Rango_Oleaje": "Altura de Olas (m)"}
)

# 9. Ajustar formato (Igual al de vientos)
cleanest_labels = np.arange(0, 360, 22.5)
fig_wave_rose.update_layout(
    polar=dict(
        radialaxis=dict(
            ticksuffix="%",
            angle=90,
            showticklabels=True
        ),
        angularaxis=dict(
            direction="clockwise",
            rotation=90,
            tickmode="array",
            tickvals=cleanest_labels,
            ticktext=[str(v) if v != 0 else "0" for v in cleanest_labels],
            tickfont=dict(size=12),
            showticklabels=True
        )
    ),
    legend=dict(
        orientation="v",
        yanchor="top", y=1.0,
        xanchor="left", x=1.1,
        bordercolor="lightgrey", 
        borderwidth=1,
        title_text="Altura de Olas (m)",
        font=dict(size=11)
    ),
    margin=dict(l=80, r=220, t=100, b=80),
    font=dict(size=12)
)

fig_wave_rose.show()